In [1]:
import os
import numpy as np
import pytorch_lightning as pl
import scanpy as sc
import pandas as pd
import torch
from torch.utils.data import DataLoader
import anndata as ad
from tqdm import tqdm

from nicheformer.models import Nicheformer
from nicheformer.data import NicheformerDataset

import time
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps



def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/batch_integrate"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            save_file_name = "record.csv"
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time / 60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            return result
        return wrapper
    return decorator

In [3]:
config = {
    'data_path': '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad', #'path/to/your/data.h5ad',  # Path to your AnnData file
    'technology_mean_path': '/home/cavin/jt/benchmark/raw_code/nicheformer/nicheformer/data/model_means/xenium_mean_script.npy', #'path/to/technology_mean.npy',  # Path to technology mean file
    'checkpoint_path': '/home/cavin/jt/benchmark/models/nicheformer/nicheformer.ckpt',  # Path to model checkpoint
    'output_path': 'nicheformer_breast_cancer_embeddings.h5ad',  # Where to save the result, it is a new h5ad
    'output_dir': '/home/cavin/jt/benchmark/experiments/middle_data',  # Directory for any intermediate outputs
    'batch_size': 16,
    'max_seq_len': 1500, 
    'aux_tokens': 30, 
    'chunk_size': 200, # to prevent OOM
    'num_workers': 4,
    'precision': 16,
    'embedding_layer': -1,  # Which layer to extract embeddings from (-1 for last layer)
    'embedding_name': 'embeddings'  # Name suffix for the embedding key in adata.obsm
}

In [4]:
model = ad.read_h5ad('/home/cavin/jt/benchmark/raw_code/nicheformer/nicheformer/data/model_means/model.h5ad')
model

AnnData object with n_obs × n_vars = 1 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology', 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'condition_id', 'tissue_type', 'library_key', 'organism', 'sex', 'niche', 'region', 'nicheformer_split', 'author_cell_type', 'batch'

In [5]:
model.var_names

Index(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419',
       'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938',
       'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084',
       'ENSG00000001167',
       ...
       'ENSMUSG00000112117', 'ENSMUSG00000112148', 'ENSMUSG00000112276',
       'ENSMUSG00000113186', 'ENSMUSG00000114028', 'ENSMUSG00000114469',
       'ENSMUSG00000115424', 'ENSMUSG00000115432', 'ENSMUSG00000115529',
       'ENSMUSG00000115965'],
      dtype='object', length=20310)

In [6]:
# Set random seed for reproducibility
pl.seed_everything(42)

# Load data
adata = ad.read_h5ad(config['data_path'])
simple_path = config['data_path']

Seed set to 42


In [8]:
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [7]:
adata.var_names

Index(['ABCC11', 'ACTA2', 'ACTG2', 'ADAM9', 'ADGRE5', 'ADH1B', 'ADIPOQ',
       'AGR3', 'AHSP', 'AIF1',
       ...
       'TUBB2B', 'TYROBP', 'UCP1', 'USP53', 'VOPP1', 'VWF', 'WARS', 'ZEB1',
       'ZEB2', 'ZNF562'],
      dtype='object', length=313)

In [9]:
adata_idlist = adata.var.ensemble_id.values
adata.var['ensembl_id'] = pd.DataFrame(adata_idlist, index=adata.var.index)
adata.var["gene_name"] = adata.var.index
adata.var.index = adata_idlist
adata.var_names.name = None
adata.var_names

Index(['ENSG00000121270', 'ENSG00000107796', 'ENSG00000163017',
       'ENSG00000168615', 'ENSG00000123146', 'ENSG00000196616',
       'ENSG00000181092', 'ENSG00000173467', 'ENSG00000169877',
       'ENSG00000204472',
       ...
       'ENSG00000137285', 'ENSG00000011600', 'ENSG00000109424',
       'ENSG00000145390', 'ENSG00000154978', 'ENSG00000110799',
       'ENSG00000140105', 'ENSG00000148516', 'ENSG00000169554',
       'ENSG00000171466'],
      dtype='object', length=313)

In [10]:
technology_mean = np.load(config['technology_mean_path'])

# format data properly with the model
combined = ad.concat([model, adata], join='outer', axis=0, fill_value=0)
aligned_combined = combined[:, model.var_names]
adata = aligned_combined[model.n_obs:, :].copy()
adata

AnnData object with n_obs × n_vars = 281780 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology', 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'condition_id', 'tissue_type', 'library_key', 'organism', 'sex', 'niche', 'region', 'nicheformer_split', 'author_cell_type', 'batch', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes'
    obsm: 'spatial'

In [11]:
adata.var_names

Index(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419',
       'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938',
       'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084',
       'ENSG00000001167',
       ...
       'ENSMUSG00000112117', 'ENSMUSG00000112148', 'ENSMUSG00000112276',
       'ENSMUSG00000113186', 'ENSMUSG00000114028', 'ENSMUSG00000114469',
       'ENSMUSG00000115424', 'ENSMUSG00000115432', 'ENSMUSG00000115529',
       'ENSMUSG00000115965'],
      dtype='object', length=20310)

In [12]:
adata.obs.nicheformer_split = 'train'
adata

AnnData object with n_obs × n_vars = 281780 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology', 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'condition_id', 'tissue_type', 'library_key', 'organism', 'sex', 'niche', 'region', 'nicheformer_split', 'author_cell_type', 'batch', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes'
    obsm: 'spatial'

As a reference, the metadata tokens are 

modality_dict = {
    'dissociated': 3,
    'spatial': 4,}

specie_dict = {
    'human': 5,
    'Homo sapiens': 5,
    'Mus musculus': 6,
    'mouse': 6,}

technology_dict = {
    "merfish": 7,
    "MERFISH": 7,
    "cosmx": 8,
    "NanoString digital spatial profiling": 8,
    "visium": 9,
    "10x 5' v2": 10,
    "10x 3' v3": 11,
    "10x 3' v2": 12,
    "10x 5' v1": 13,
    "10x 3' v1": 14,
    "10x 3' transcription profiling": 15, 
    "10x transcription profiling": 15,
    "10x 5' transcription profiling": 16,
    "CITE-seq": 17, 
    "Smart-seq v4": 18,
}

In [13]:
# Change accordingly

adata.obs['modality'] = 4 # spatial
adata.obs['specie'] = 5 # human
adata.obs['assay'] = 8 #cosmx
adata

AnnData object with n_obs × n_vars = 281780 × 20310
    obs: 'soma_joinid', 'is_primary_data', 'dataset_id', 'donor_id', 'assay', 'cell_type', 'development_stage', 'disease', 'tissue', 'tissue_general', 'specie', 'technology', 'dataset', 'x', 'y', 'assay_ontology_term_id', 'sex_ontology_term_id', 'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'condition_id', 'tissue_type', 'library_key', 'organism', 'sex', 'niche', 'region', 'nicheformer_split', 'author_cell_type', 'batch', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes', 'modality'
    obsm: 'spatial'

In [14]:
adata.var_names

Index(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419',
       'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938',
       'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084',
       'ENSG00000001167',
       ...
       'ENSMUSG00000112117', 'ENSMUSG00000112148', 'ENSMUSG00000112276',
       'ENSMUSG00000113186', 'ENSMUSG00000114028', 'ENSMUSG00000114469',
       'ENSMUSG00000115424', 'ENSMUSG00000115432', 'ENSMUSG00000115529',
       'ENSMUSG00000115965'],
      dtype='object', length=20310)

In [15]:
# Create dataset
dataset = NicheformerDataset(
    adata=adata,
    technology_mean=technology_mean,
    split='train',
    max_seq_len=1500,
    aux_tokens=config.get('aux_tokens', 30),
    chunk_size=config.get('chunk_size', 1000),
    metadata_fields={'obs': ['modality', 'specie', 'assay']}
)

# Create dataloader
dataloader = DataLoader(
    dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config.get('num_workers', 4),
    pin_memory=True
)

100%|██████████| 1409/1409 [05:27<00:00,  4.31it/s]


In [16]:
@measure_resources(cuda_id=0, task_des="Nicheformer run xenium human breast cancer")
def get_emb():
    # Load pre-trained model
    model = Nicheformer.load_from_checkpoint(checkpoint_path=config['checkpoint_path'], strict=False,weights_only=False)
    model.eval()  # Set to evaluation mode

    # Configure trainer
    trainer = pl.Trainer(
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        default_root_dir=config['output_dir'],
        precision=config.get('precision', 32),
    )
    print("Extracting embeddings...")
    embeddings = []
    device = model.embeddings.weight.device

    with torch.no_grad():
        for batch in tqdm(dataloader):
            # Move batch to device
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                    for k, v in batch.items()}

            # Get embeddings from the model
            emb = model.get_embeddings(
                batch=batch,
                layer=config.get('embedding_layer', -1)  # Default to last layer
            )
            embeddings.append(emb.cpu().numpy())


    # Concatenate all embeddings
    embeddings = np.concatenate(embeddings, axis=0)
    return embeddings

In [17]:
emb = get_emb()

/home/cavin/anaconda3/envs/nicheformer_env/lib/python3.9/site-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/cavin/anaconda3/envs/nicheformer_env/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboar

Extracting embeddings...


100%|██████████| 17612/17612 [2:10:03<00:00,  2.26it/s] 



RESOURCE USAGE REPORT: get_emb
Timestamp                   : 2026-03-05 13:29:38
Target GPU ID               : 0
Execution Time (Minutes)    : 130.1393
Execution Time (Seconds)    : 7808.36
CPU Peak Memory Usage (GB)  : 5.1503
GPU Peak Allocated Mem (GB) : 9.0133
GPU Peak Cached Mem (GB)    : 11.1309
CUDA Available              : True

文件已存在，数据已成功追加至末尾。


In [18]:
emb.shape

(281780, 512)

In [19]:
emb_df = pd.DataFrame(
    emb, 
    index=adata.obs_names,
    columns=[f"nicheformer_dim_{i}" for i in range(emb.shape[1])]
)
save_path = "/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/nicheformer_human_breast_cancer_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")

In [20]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [21]:
adata.obsm["X_nicheformer"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_nicheformer'